# Sales Data Analysis

Region: **{region}** &nbsp;|&nbsp; Minimum sales threshold: **{min_sales}**

This notebook filters a deterministic local sales dataset by region and minimum revenue threshold, then summarizes revenue by product and renders a chart.

In [1]:
# parameters
region = "North"
min_sales = 5000


In [2]:
# Parameters
PAPERMILL_INPUT_PATH = "K:\\Capstone\\uploads\\15\\input.ipynb"
PAPERMILL_OUTPUT_PATH = "K:\\Capstone\\runs\\15\\output.ipynb"
region = "North"
min_sales = 5000


In [3]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
regions = ['North', 'South', 'East', 'West']
products = ['Widget', 'Gadget', 'Gizmo', 'Doohickey', 'Thingamajig']
salespeople = ['Alice', 'Bob', 'Carol', 'Dan', 'Eve', 'Frank', 'Grace', 'Heidi']

n = 1200
df = pd.DataFrame({
    'date': pd.date_range('2025-01-01', periods=n, freq='D'),
    'region': rng.choice(regions, size=n, p=[0.30, 0.25, 0.25, 0.20]),
    'product': rng.choice(products, size=n),
    'salesperson': rng.choice(salespeople, size=n),
    'units': rng.integers(1, 50, size=n),
})
df['revenue'] = df['units'] * rng.integers(100, 900, size=n)

print(f'Generated {len(df)} sales records (seed=42)')
print(f'Regions present: {sorted(df.region.unique())}')


Generated 1200 sales records (seed=42)
Regions present: ['East', 'North', 'South', 'West']


In [4]:
import os
out_dir = os.path.dirname(os.path.abspath(PAPERMILL_OUTPUT_PATH))

filtered = df[(df['region'] == region) & (df['revenue'] >= min_sales)]

summary = (filtered
    .groupby('product')
    .agg(total_revenue=('revenue', 'sum'),
         total_units=('units', 'sum'),
         num_sales=('revenue', 'size'))
    .sort_values('total_revenue', ascending=False)
    .reset_index())

print(f'\nFilter: region={region}, min_sales={min_sales}')
print(f'Rows surviving filter: {len(filtered)} of {len(df)}')
print(f'Total revenue: ${filtered["revenue"].sum():,.2f}')
print('\nRevenue summary by product:')
print(summary.to_string(index=False))

summary_path = os.path.join(out_dir, 'summary.json')
summary.to_json(summary_path, orient='records', indent=2)
print(f'\nWrote {summary_path}')



Filter: region=North, min_sales=5000
Rows surviving filter: 267 of 1200
Total revenue: $4,215,655.00

Revenue summary by product:
    product  total_revenue  total_units  num_sales
     Gadget         962027         1892         62
Thingamajig         895246         1638         53
      Gizmo         807943         1669         51
  Doohickey         791946         1442         50
     Widget         758493         1675         51

Wrote K:\Capstone\runs\15\summary.json


In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(summary['product'], summary['total_revenue'],
              color=['#4C78A8', '#E45756', '#79706E', '#54A24B', '#Eeca3b'])
ax.set_title(f'Sales Revenue by Product\n(region={region}, min_sales={min_sales})')
ax.set_xlabel('Product')
ax.set_ylabel('Total Revenue ($)')
ax.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, summary['total_revenue']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'${val:,.0f}', ha='center', va='bottom', fontsize=9)
fig.tight_layout()
chart_path = os.path.join(out_dir, 'chart.png')
fig.savefig(chart_path, dpi=110)
print(f'Wrote {chart_path}')
plt.close(fig)


Wrote K:\Capstone\runs\15\chart.png
